# 05: Building a Support Ticket RAG System with Chroma DB & OpenAI

Welcome to **Notebook 05** of the **Learn Python** series! This step-by-step hands-on guide covers building a **Retrieval-Augmented Generation (RAG)** application using **Chroma DB** and **OpenAI**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohit-Saini-Sfdc/learn-python/blob/main/05_chroma_db_rag_mastery.ipynb)

---

### 📌 Practical Use Case
In software applications, customer support teams frequently handle repetitive questions and technical issues. In this tutorial, we build an **AI Support Ticket Assistant**:
1. When a **new support ticket** arrives, we convert its description into a vector embedding.
2. We query **Chroma DB** to retrieve historical tickets with the highest **Cosine Similarity**.
3. We feed the retrieved historical resolutions as context to an **OpenAI LLM (`gpt-4o-mini`)** to automatically generate a suggested resolution for the support agent.

---

### 🎯 What You Will Learn
1. **Dependencies & Setup**: Installing Chroma DB and OpenAI SDK in Google Colab.
2. **Support Ticket Dataset**: Structuring text and metadata for RAG.
3. **Chroma DB Initialization**: Configuring collection distance metrics (**Cosine Similarity**).
4. **Vector Embeddings**: Indexing tickets with OpenAI's `text-embedding-3-small`.
5. **Similarity Search**: Querying top-K matching tickets using Cosine Distance.
6. **RAG Pipeline**: Constructing prompts and generating LLM responses.
7. **Metadata Filtering & Management**: Filtering by category/tags and updating ticket collections.

Let's get started! 🎯


## Step 1: Install Dependencies & Setup OpenAI API Key

First, let's install the required Python packages:
- `chromadb`: The vector database for storing and searching embeddings.
- `openai`: OpenAI API client for generating embeddings and LLM completions.
- `tiktoken`: Tokenizer utility for OpenAI models.


In [ ]:
# Install required packages
!pip install -q chromadb openai tiktoken python-dotenv


Now, let's set up the OpenAI API Key. In Google Colab, you can store your API Key securely in **Secrets** (the key icon on the left panel named `OPENAI_API_KEY`), or enter it interactively when prompted.


In [ ]:
import os
import getpass

# Try retrieving from Google Colab Secrets (if running in Colab)
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = None

# If not found in Secrets, prompt interactively
if not api_key:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        api_key = getpass.getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = api_key
print("✅ OpenAI API Key set successfully!")


## Step 2: Define the Support Ticket Dataset

Let's create a dataset of historical support tickets. Each ticket includes:
- `ticket_id`: Unique identifier.
- `title`: Short title of the issue.
- `description`: Detailed problem statement submitted by the customer.
- `resolution`: Verified solution provided by senior support engineers.
- `category`: Ticket topic (`Authentication`, `Database`, `Billing`, `Performance`, `API`).
- `priority`: Priority level (`High`, `Medium`, `Low`).

In RAG, we concatenate the `title`, `description`, and `resolution` into a combined document text for embedding, while keeping attributes like `category` and `priority` in **metadata** for filtered queries.


In [ ]:
# Sample historical support tickets dataset
historical_tickets = [
    {
        "ticket_id": "TICK-101",
        "title": "SSO Login Failure with Okta",
        "description": "Users receive HTTP 403 Forbidden error when logging in via Okta SAML SSO after recent certificate renewal.",
        "resolution": "Re-uploaded the updated Okta IdP metadata XML file into Settings > Single Sign-On and cleared user session cache.",
        "category": "Authentication",
        "priority": "High"
    },
    {
        "ticket_id": "TICK-102",
        "title": "Database Query Timeout on Large Reports",
        "description": "Exporting CSV reports containing more than 50,000 rows fails with a 504 Gateway Timeout error after 30 seconds.",
        "resolution": "Added composite index on (created_at, account_id) in Postgres and increased NGINX proxy_read_timeout to 120s.",
        "category": "Database",
        "priority": "High"
    },
    {
        "ticket_id": "TICK-103",
        "title": "Duplicate Billing Charge on Subscription Upgrade",
        "description": "Customer was charged twice when upgrading from Pro to Enterprise tier in the middle of a billing cycle.",
        "resolution": "Issued a prorated refund of $49 via Stripe dashboard and updated billing webhook handler to acquire idempotency lock.",
        "category": "Billing",
        "priority": "Medium"
    },
    {
        "ticket_id": "TICK-104",
        "title": "API Rate Limit 429 Errors on Batch Endpoint",
        "description": "Automated scripts making 200 requests/minute to /api/v1/batch-sync receive 429 Too Many Requests status code.",
        "resolution": "Configured Redis Token Bucket size from 100 to 500 requests/min for verified Enterprise API keys.",
        "category": "API",
        "priority": "Medium"
    },
    {
        "ticket_id": "TICK-105",
        "title": "Password Reset Email Not Received",
        "description": "Users with @company.org email domains do not receive the password reset link email.",
        "resolution": "Whitelisted email sending domain in SendGrid SPF/DKIM records and unblocked @company.org domain in bounce list.",
        "category": "Authentication",
        "priority": "Low"
    },
    {
        "ticket_id": "TICK-106",
        "title": "High Memory Consumption in Background Worker",
        "description": "Sidekiq background worker pods crash periodically due to Out-Of-Memory (OOM) killer during PDF generation.",
        "resolution": "Switched PDF renderer to stream output directly to S3 bucket rather than loading full document into Ruby heap memory.",
        "category": "Performance",
        "priority": "High"
    }
]

print(f"Loaded {len(historical_tickets)} historical support tickets.")


## Step 3: Initialize Chroma DB & Configure Cosine Similarity

### 💡 Understanding Distance Metrics in Vector Databases
When Chroma DB calculates similarity between vectors, it uses a distance metric:
- **Cosine Distance (`cosine`)**: Measures the **angle** between two vectors, ignoring magnitude. Ranges from `0` (identical direction / highest similarity) to `2` (opposite direction). **Best for text embeddings.**
- **L2 / Euclidean Distance (`l2`)**: Measures direct straight-line distance between points.
- **Inner Product (`ip`)**: Measures dot product.

We configure our Chroma DB collection with `{"hnsw:space": "cosine"}` to use **Cosine Distance**.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize an in-memory Chroma DB client
chroma_client = chromadb.EphemeralClient()

# Set up OpenAI Embedding Function using text-embedding-3-small
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small"
)

# Create or get Chroma collection with Cosine Similarity metric
collection_name = "support_tickets_rag"

# Delete collection if it already exists for a clean run
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

ticket_collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"} # Explicitly configure Cosine Similarity
)

print(f"✅ Created Chroma DB collection '{collection_name}' configured with Cosine Similarity!")


## Step 4: Index Support Tickets into Chroma DB

Now we prepare the support ticket data for insertion into Chroma DB:
1. **`documents`**: The text that Chroma will convert into embeddings using `text-embedding-3-small`. We format this into a structured document text containing Title, Description, and Resolution.
2. **`metadatas`**: Dictionary of metadata attributes (`ticket_id`, `category`, `priority`, `resolution`).
3. **`ids`**: Unique string IDs (`TICK-101`, `TICK-102`, etc.).


In [ ]:
documents = []
metadatas = []
ids = []

for t in historical_tickets:
    # Build text representation for vector embedding
    doc_text = f"Title: {t['title']}\nCategory: {t['category']}\nProblem: {t['description']}\nResolution: {t['resolution']}"
    
    documents.append(doc_text)
    metadatas.append({
        "ticket_id": t["ticket_id"],
        "title": t["title"],
        "category": t["category"],
        "priority": t["priority"],
        "resolution": t["resolution"]
    })
    ids.append(t["ticket_id"])

# Add documents to Chroma collection (embeddings are automatically computed via OpenAI embedding function)
ticket_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f" Successfully indexed {ticket_collection.count()} support tickets into Chroma DB!")


## Step 5: Querying Chroma DB with Cosine Similarity

When a customer submits a **new ticket**, we query Chroma DB to find the K most similar past tickets.

Let's test with a new ticket question:
> *"Users report 500 error when exporting large CSV files from our dashboard."*

Chroma will:
1. Embed the query text using `text-embedding-3-small`.
2. Compute **Cosine Distance** against all stored ticket vectors.
3. Return the closest matching historical tickets.


In [ ]:
new_query = "Users receive 500 server error when trying to export large CSV files"

# Perform Cosine Similarity Search
results = ticket_collection.query(
    query_texts=[new_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"🔍 Query: '{new_query}'\n")
print("Top Relevant Historical Tickets Found:\n" + "="*50)

for i in range(len(results["ids"][0])):
    ticket_id = results["ids"][0][i]
    meta = results["metadatas"][0][i]
    distance = results["distances"][0][i]
    
    # Cosine Distance -> Cosine Similarity conversion: similarity = 1 - distance
    similarity_score = 1 - distance
    
    print(f"Rank {i+1}: [{ticket_id}] {meta['title']}")
    print(f"   Category: {meta['category']} | Priority: {meta['priority']}")
    print(f"   Cosine Distance: {distance:.4f} (Similarity: {similarity_score:.4f})")
    print(f"   Past Resolution: {meta['resolution']}\n")


## Step 6: Full RAG Pipeline using OpenAI LLM

Now we combine **Retrieval** (Chroma DB similarity search) and **Generation** (OpenAI `gpt-4o-mini`) to answer the new ticket.

### RAG Steps:
1. **Retrieve**: Find top K relevant tickets from Chroma DB.
2. **Build Prompt**: Formulate a system prompt containing the retrieved context + the new ticket question.
3. **Generate**: Call OpenAI `chat.completions.create` to generate a comprehensive, actionable response for the support agent.


In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def solve_support_ticket_rag(new_ticket_description: str, top_k: int = 2) -> str:
    # 1. Retrieve relevant historical tickets
    search_results = ticket_collection.query(
        query_texts=[new_ticket_description],
        n_results=top_k,
        include=["metadatas", "distances"]
    )
    
    # 2. Build context string from retrieved resolutions
    context_blocks = []
    for i in range(len(search_results["ids"][0])):
        meta = search_results["metadatas"][0][i]
        dist = search_results["distances"][0][i]
        context_blocks.append(
            f"--- Historical Ticket Reference #{i+1} ---\n"
            f"Ticket ID: {meta['ticket_id']}\n"
            f"Title: {meta['title']}\n"
            f"Category: {meta['category']}\n"
            f"Resolution: {meta['resolution']}\n"
            f"Similarity Score: {1 - dist:.3f}"
        )
    
    context_str = "\n\n".join(context_blocks)
    
    # 3. Construct System and User Prompts
    system_prompt = (
        "You are an expert AI Customer Support Assistant for a SaaS application.\n"
        "Your task is to analyze new support ticket questions and suggest an accurate, "
        "step-by-step resolution based ONLY on the provided historical support ticket context.\n"
        "If the context contains relevant solutions, synthesize a clear recommendation.\n"
        "Be professional, concise, and structured in your answer."
    )

    user_prompt = (
        f"NEW SUPPORT TICKET QUESTION:\n{new_ticket_description}\n\n"
        f"RELEVANT HISTORICAL TICKETS & RESOLUTIONS:\n{context_str}\n\n"
        f"SUGGESTED ACTION PLAN & RESOLUTION:"
    )

    # 4. Generate response using OpenAI LLM
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )
    
    return response.choices[0].message.content


### Testing the RAG Pipeline with New Ticket Questions

Let's test our RAG assistant on 2 different support tickets!


In [ ]:
# Test Case 1: Database & Export Timeout
test_ticket_1 = "Customer reports that exporting large analytics datasets times out after 30 seconds with 504 error."

print("--------------------------------------------------")
print("📥 NEW TICKET 1:", test_ticket_1)
print("--------------------------------------------------")
answer_1 = solve_support_ticket_rag(test_ticket_1, top_k=2)
print("🤖 AI SUGGESTED RESOLUTION:\n")
print(answer_1)


In [ ]:
# Test Case 2: Authentication / SSO Issue
test_ticket_2 = "Our team members cannot log in with Okta single sign on. They get a 403 error page."

print("--------------------------------------------------")
print("📥 NEW TICKET 2:", test_ticket_2)
print("--------------------------------------------------")
answer_2 = solve_support_ticket_rag(test_ticket_2, top_k=2)
print("🤖 AI SUGGESTED RESOLUTION:\n")
print(answer_2)


## Step 7: Advanced Chroma DB Features (Filtering & Management)

### 1. Metadata Filtering (`where` clause)
Chroma DB allows you to filter vector searches using metadata fields (e.g., search ONLY within `Authentication` category).


In [ ]:
# Query with metadata filter for Authentication tickets only
filtered_results = ticket_collection.query(
    query_texts=["login problem"],
    n_results=2,
    where={"category": "Authentication"} # Metadata filter
)

print("Filtered Search Results (Category == 'Authentication'):")
for meta in filtered_results["metadatas"][0]:
    print(f"- [{meta['ticket_id']}] {meta['title']} ({meta['category']})")


### 2. Updating / Upserting Tickets in Chroma DB
If a historical ticket's resolution is updated, you can update it in Chroma DB using `upsert()`:


In [ ]:
# Upsert an updated ticket resolution
ticket_collection.upsert(
    ids=["TICK-105"],
    documents=["Title: Password Reset Email Not Received\nCategory: Authentication\nProblem: Password reset emails blocked\nResolution: Whitelisted SendGrid IP and updated DKIM 2048-bit keys."],
    metadatas=[{
        "ticket_id": "TICK-105",
        "title": "Password Reset Email Not Received",
        "category": "Authentication",
        "priority": "Medium",
        "resolution": "Whitelisted SendGrid IP and updated DKIM 2048-bit keys."
    }]
)

print("✅ Successfully updated TICK-105 in Chroma DB!")


---
## 🎓 Summary & Recap

Congratulations! 🎉 You have built a complete, production-ready RAG application for Support Tickets using Chroma DB and OpenAI!

### Key Takeaways:
1. **Chroma DB Ephemeral / Persistent Client**: Quick to set up and easy to scale.
2. **Cosine Distance (`hnsw:space: cosine`)**: Optimal for semantic text similarity.
3. **OpenAI Embedding Integration**: `text-embedding-3-small` generates high-quality semantic representations.
4. **Context Injection**: RAG retrieves relevant past resolutions and injects them into `gpt-4o-mini` prompts for accurate AI answers.
5. **Metadata Filtering**: Enables hybrid search combining vector similarity with exact category/priority filters.

### 🚀 Next Steps to Extend this Project:
- Try persistent storage with `chromadb.PersistentClient(path="./chroma_db")`.
- Add Hybrid Search (combining BM25 text search with Cosine Vector search).
- Integrate a chat UI like Streamlit or Gradio to deploy your AI Support Assistant!
